In [37]:
import sys
sys.path.append("..")

In [38]:
from pathlib import Path

import pandas as pd

from src.tokenizer.corpus import Corpus
from src.tokenizer.statistics import Statistics
from src.tokenizer.candidates import CandidateExtractor
from src.tokenizer.normalizer import Normalizer

In [39]:
DATA_PATH = Path("../data/text_corpus.csv")

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8"
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   comment  10000 non-null  str  
dtypes: str(1)
memory usage: 78.3 KB


In [40]:
print(df.shape)
print(df.columns.tolist())

(10000, 1)
['comment']


In [41]:
texts = (
    df["comment"]
    .dropna()
    .tolist()
)

corpus = Corpus(texts)

print("Documents:", len(corpus))

Documents: 10000


In [42]:
for i, text in enumerate(corpus):

    print(text)

    if i == 4:
        break

доступ в Интернет (FTTx) отсутствует. Причина - проблема абонентского оборудования.
OTT отсутствует. Причина - воздействие третьей стороны.
PSTN отсутствует. Причина - неисправность оборудования провайдера.
По услуге Интернет есть проблемы. Причина - проблема абонентского оборудования.
CTV. не работают приложения. Причина - авария оборудования.


# частоты

In [43]:
stats = Statistics(
    min_n=2,
    max_n=8
)

stats.fit(corpus)

print(
    "Unique ngrams:",
    len(stats.ngram_freq)
)

Unique ngrams: 9831


In [44]:
print(
    "Unique words:",
    len(stats.word_freq)
)

Unique words: 80


In [45]:
for word, freq in stats.word_freq.most_common(100):

    print(
        f"{freq:6d}",
        word
    )

  8524 причина
  7976 -
  6357 оборудования
  3552 провайдера
  3350 отсутствует
  3060 по
  2898 неисправность
  2797 проблема
  2655 интернет
  1961 авария
  1873 услуге
  1793 есть
  1793 проблемы
  1650 на
  1605 абонента
  1461 шпд
  1398 в
  1382 абонентского
  1350 доступ
  1187 неверная
  1187 конфигурация
  1068 плохое
  1068 качество
  1027 нет
   992 зарегистрировано
   965 ott
   848 не
   848 работают
   848 приложения
   719 соединение
   669 соединения
   658 стороне
   641 сети
   582 триплплей
   551 падает
   551 сессия
   540 все-в-одном
   536 вип
   494 цифровое
   494 тв
   490 ctv
   450 fttx
   427 iot
   421 gpon
   407 xpon
   406 воздействие
   406 третьей
   406 стороны
   386 pon
   376 dhcp
   348 аналоговая
   348 телефония
   329 pstn
   325 авторизации
   295 xdsl
   224 выполняется
   202 wifi
   186 нулевой
   186 баланс
   179 беспроводной
   174 wi-fi
   160 ошибка
   158 tdm
   155 витая
   155 пара
   154 сайт
   154 запрещен
   154 регулятором
  

In [46]:
for word, freq in stats.word_document_freq.most_common(100):

    print(
        f"{freq:6d}",
        word
    )

  8524 причина
  7976 -
  6357 оборудования
  3552 провайдера
  3350 отсутствует
  2898 неисправность
  2829 по
  2797 проблема
  2655 интернет
  1961 авария
  1873 услуге
  1793 есть
  1793 проблемы
  1605 абонента
  1581 на
  1439 шпд
  1398 в
  1382 абонентского
  1350 доступ
  1187 неверная
  1187 конфигурация
  1068 качество
  1068 плохое
  1027 нет
   992 зарегистрировано
   965 ott
   848 не
   848 приложения
   848 работают
   719 соединение
   669 соединения
   658 стороне
   641 сети
   582 триплплей
   551 сессия
   551 падает
   540 все-в-одном
   536 вип
   494 цифровое
   494 тв
   490 ctv
   450 fttx
   427 iot
   421 gpon
   407 xpon
   406 стороны
   406 третьей
   406 воздействие
   386 pon
   376 dhcp
   348 телефония
   348 аналоговая
   329 pstn
   325 авторизации
   295 xdsl
   224 выполняется
   202 wifi
   186 нулевой
   186 баланс
   179 беспроводной
   174 wi-fi
   160 ошибка
   158 tdm
   155 пара
   155 витая
   154 сайт
   154 запрещен
   154 регулятором
  

In [47]:
terms = [
    "gpon",
    "pon",
    "fttx",
    "wifi",
    "internet",
    "интернет",
    "оборудование",
    "соединение",
    "авторизация"
]


for term in terms:

    print("\n====", term)

    print(
        "freq:",
        stats.word_freq[term.lower()]
    )

    print(
        "documents:",
        stats.word_document_freq[term.lower()]
    )


==== gpon
freq: 421
documents: 421

==== pon
freq: 386
documents: 386

==== fttx
freq: 450
documents: 450

==== wifi
freq: 202
documents: 202

==== internet
freq: 0
documents: 0

==== интернет
freq: 2655
documents: 2655

==== оборудование
freq: 0
documents: 0

==== соединение
freq: 719
documents: 719

==== авторизация
freq: 0
documents: 0


In [48]:
term = "оборудования"


print(
    "LEFT:"
)

for word, count in stats.left_context[term].most_common(10):

    print(
        count,
        word
    )


print(
    "\nRIGHT:"
)

for word, count in stats.right_context[term].most_common(10):

    print(
        count,
        word
    )

LEFT:
2898 неисправность
1415 проблема
1382 абонентского
662 авария

RIGHT:
2894 провайдера
1419 абонента
52 выполняется
27 передано
21 ожидается


In [49]:
for phrase, freq in stats.phrase_freq.most_common(100):

    print(
        f"{freq:6d}",
        phrase
    )

  7976 причина -
  3191 отсутствует причина
  2968 отсутствует причина -
  2898 неисправность оборудования
  2894 оборудования провайдера
  2445 - неисправность
  2445 причина - неисправность
  2445 - неисправность оборудования
  2360 - проблема
  2360 причина - проблема
  1873 по услуге
  1793 есть проблемы
  1793 проблемы причина
  1793 есть проблемы причина
  1691 проблемы причина -
  1650 - авария
  1650 причина - авария
  1479 неисправность оборудования провайдера
  1419 оборудования абонента
  1419 неисправность оборудования абонента
  1415 проблема оборудования
  1415 проблема оборудования провайдера
  1382 проблема абонентского
  1382 абонентского оборудования
  1382 проблема абонентского оборудования
  1350 доступ в
  1350 в интернет
  1350 доступ в интернет
  1206 - проблема оборудования
  1187 неверная конфигурация
  1187 конфигурация по
  1187 неверная конфигурация по
  1154 - проблема абонентского
  1068 плохое качество
  1068 качество причина
  1068 плохое качество причин

# кандидаты

In [50]:
extractor = CandidateExtractor(
    normalizer=Normalizer(),
    min_freq=50
)

In [51]:
candidates = extractor.extract(
    stats
)

In [52]:
for item in candidates[:100]:

    print(
        item
    )

('неисправность оборудования', 5343, 15112.286127518895)
('есть проблемы', 3586, 10142.739669339839)
('оборудования провайдера', 2894, 8185.468099015475)
('неисправность оборудования провайдера', 1479, 7685.109433183109)
('проблема оборудования', 2621, 7413.3074939597645)
('неисправность оборудования абонента', 1419, 7373.340287820711)
('проблема оборудования провайдера', 1415, 7352.555678129885)
('проблема абонентского оборудования', 1382, 7181.082648180565)
('проблема абонентского', 2536, 7172.891188356339)
('доступ в интернет', 1350, 7014.805770653953)
('неверная конфигурация', 2207, 6242.338664314842)
('неверная конфигурация по', 1187, 6167.832925752772)
('отсутствует', 6159, 6159.0)
('плохое качество', 2136, 6041.520338457863)
('неисправность', 4890, 4890.0)
('проблема', 4720, 4720.0)
('не работают приложения', 848, 4406.337254455224)
('работают приложения', 1544, 4367.091480608118)
('оборудования абонента', 1419, 4013.538090014844)
('абонентского оборудования', 1382, 3908.8862863

# Получается интересное разделение

## Я (chatGPT) бы ввел три класса токенов.

### 1. Базовые термины
* авария
* неисправность
* качество
* отсутствует
* соединение
* авторизация
* оборудование

Они описывают основные понятия предметной области.

### 2. Составные термины
* авария оборудования
* авария сети
* неисправность оборудования
* не работают приложения
* нет соединения
* падает сессия

Они уточняют смысл базовых терминов.

### 3. Конкретные именованные сущности
* GPON
* FTTx
* xDSL
* WiFi
* DHCP
* CTV
* OTT
* Интернет
* ШПД

Это технологии, сервисы и протоколы.

## Идея для Evaluator

Вместо того чтобы пытаться оставить только одну "лучшую" форму, можно начислять баллы по разным критериям:

* Критерий:	Что поощряет
* Частота:	Часто встречающиеся термины
* PMI:	Устойчивые словосочетания
* Информационный прирост:	Фраза добавляет смысл по сравнению с подфразами
* Тип токена:	Базовый термин, составной термин, сущность
* Дискриминативность:	Насколько токен помогает различать комментарии

Последний пункт, на мой взгляд, особенно перспективен. Если токен вроде `авария` помогает отделить один класс обращений от других, он ценен независимо от того, существует ли более длинная фраза `авария оборудования`. Это хорошо согласуется с вашей конечной целью — улучшить BoW-представление для кластеризации комментариев, а не просто собрать список красивых словосочетаний. Именно дискриминативная способность токена может стать одним из ключевых критериев качества словаря.

# Context-copy-past-chatGPT

Проект: построение специализированного Vocabulary Builder / Tokenizer для коротких CRM/NOC комментариев службы технической поддержки телеком-оператора.

Цель:
Не использовать готовые токенизаторы (BPE, WordPiece и т.п.) и не использовать готовые эмбеддинги, а автоматически построить предметно-ориентированный словарь токенов по статистике конкретного корпуса.

Конечная цель:
Tokenizer → BoW → косинусная близость → качественная кластеризация CRM комментариев.

Основная идея:
Оптимизируется не языковая модель, а качество семантических признаков для BoW.

Структура проекта:

src/
    generator.py                 # синтетический генератор корпуса
    tokenizer/
        corpus.py
        statistics.py
        vocabulary.py
        normalizer.py
        ...

notebooks/
    все эксперименты выполняются только из ноутбуков

data/
    text_corpus.csv

----------------------------------------
Генератор корпуса

CRMGenerator полностью работает.

Исправлено:

- если alias отсутствует,
  generator автоматически берет title
  из соответствующего yaml.

Никакого дублирования aliases.yaml нет.

Корпус теперь полностью русифицирован
(кроме осознанно оставленных технологий GPON, WiFi, OTT, xDSL и т.п.)

----------------------------------------
Pipeline

CRMGenerator
      ↓
Corpus
      ↓
Statistics
      ↓
Normalizer
      ↓
CandidateExtractor
      ↓
Vocabulary
      ↓
Tokenizer
      ↓
BoW
      ↓
Evaluator

----------------------------------------
Statistics

Сейчас считает

char_freq

word_freq

doc_freq

left_context

right_context

phrase_freq

phrase_freq содержит все биграммы
и триграммы.

Добавлено:

for n in [2,3]:
    ...

Это уже работает.

----------------------------------------
Normalizer

Файл

src/tokenizer/normalizer.py

делает

lower()

удаление лишних пробелов

удаление

"Причина -"

"-"

"По услуге"

и пр.

НО

Normalizer НЕ удаляет
семантические слова.

----------------------------------------
CandidateExtractor

Уже переписан.

Теперь работает в два прохода.

1)

нормализует кандидатов

агрегирует одинаковые

Counter()

2)

строит score

score =

freq * len(words)**1.5

После агрегирования

например

неисправность оборудования

2898

+

2445

↓

5343

----------------------------------------
Полученные результаты

Очень хорошо выделяются

неисправность оборудования

неисправность оборудования провайдера

неисправность оборудования абонента

проблема оборудования

проблема абонентского оборудования

доступ в интернет

не работают приложения

нет соединения

соединение отсутствует

авария сети

авария оборудования

воздействие третьей стороны

падает сессия

цифровое ТВ

аналоговая телефония

нет DHCP

ошибка авторизации

и т.д.

----------------------------------------
Главный вывод

НЕ считать однословные термины мусором.

Например

авария

качество

неисправность

проблема

отсутствует

являются важнейшими
семантическими признаками.

Они должны оставаться в словаре.

Длинная фраза НЕ заменяет короткую.

Например

авария

авария сети

авария оборудования

авария на стороне провайдера

должны существовать одновременно.

----------------------------------------
Следующая цель

Не бороться за "самую длинную фразу".

Строить словарь
семантических единиц
разных уровней.

Получается почти онтология предметной области.

----------------------------------------
Следующие модули

CandidateExtractor уже достаточно хороший.

Следующий серьезный модуль —

Vocabulary

который будет

ранжировать

фильтровать

строить финальный словарь.

После этого

Tokenizer

должен выполнять longest-match tokenization
по построенному словарю.

----------------------------------------
Дальнейшая цель

Evaluator.

Но не через качество самого словаря.

Настоящий критерий качества —

качество кластеризации CRM комментариев после BoW.

То есть

Vocabulary

оценивается косвенно

через улучшение структуры пространства документов.

Это фактически задача оптимизации словаря под последующую кластеризацию, а не под языковое моделирование.

----------------------------------------
Архитектурная идея

Мы НЕ строим очередной BPE.

Мы строим статистический терминологический словарь предметной области.

Это ближе к Phrase Mining + Vocabulary Learning, чем к классическим субсловным токенизаторам.